<a href="https://colab.research.google.com/github/cz2112/hw/blob/main/Notebooks_cz/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [5]:
# Now let's compute self attention in matrix form
def graph_attention(X, omega, beta, phi, A):

    N = X.shape[1]

    # 1. Compute X_prime
    # Shape of X_prime will be (D, N)
    X_prime = np.matmul(omega, X) + beta

    # 2. Compute S
    # We will build the score matrix S where S[j, i] holds the score
    # for target node i attending to source node j.
    S = np.zeros((N, N))

    for i in range(N): # i is our target node (column index)
        for j in range(N): # j is our source node (row index)
            # Concatenate the feature vector of target i and source j
            # Both slices are (D, 1), so concatenated is (2D, 1)
            concat_features = np.concatenate((X_prime[:, i:i+1], X_prime[:, j:j+1]), axis=0)

            # Multiply by phi transpose to get a scalar score
            # phi.T is (1, 2D). Result is (1, 1). Extract the scalar.
            S[j, i] = np.matmul(phi.T, concat_features)[0, 0]

    # 3. Apply the mask
    # A node attends to its neighbors and itself, so we add the identity matrix
    valid_connections = A + np.eye(N)

    # Using boolean array indexing to set non-connections to a large negative number
    S[valid_connections == 0] = -1e20

    # 4. Run the softmax function to compute the attention values
    # Since S[j, i] is source j for target i, softmax over columns correctly
    # normalizes the weights for each target node independently.
    attention_weights = softmax_cols(S)

    # 5. Postmultiply X' by the attention values
    # X_prime is (D, N), attention_weights is (N, N).
    # Resulting matrix will be (D, N).
    aggregated_features = np.matmul(X_prime, attention_weights)

    # 6. Apply the ReLU function
    output = ReLU(aggregated_features)

    return output

In [6]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[1.983 1.904 0.228 1.974 1.259 1.251 1.251 1.251]
 [1.404 1.384 0.168 1.38  5.333 5.367 5.392 5.367]
 [0.378 0.342 0.    0.381 1.112 1.12  1.125 1.12 ]
 [0.    0.    0.    0.    1.079 1.097 1.108 1.097]]


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.


In [7]:
# TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1
# that respects the geometry of the graph and has zero attention between non-neighboring nodes
# by combining figures 13.12a and 13.12b.

def masked_dot_product_attention(X, W_q, W_k, W_v, A):
    N = X.shape[1]

    # 1. Compute Queries (Q), Keys (K), and Values (V)
    # Output shapes are all (D, N)
    Q = np.matmul(W_q, X)
    K = np.matmul(W_k, X)
    V = np.matmul(W_v, X)

    # 2. Calculate raw attention scores
    # We want S[j, i] to be the dot product of key j and query i.
    # K.T is (N, D) and Q is (D, N) -> S is (N, N)
    S = np.matmul(K.T, Q)

    # 3. Restrict attention to graph geometry
    # Allow attention only where there is an edge, plus self-loops
    allowed_edges = A + np.eye(N)

    # Set impossible edges to negative infinity (practically)
    S[allowed_edges == 0] = -1e20

    # 4. Normalize the scores using softmax over columns
    alpha = softmax_cols(S)

    # 5. Aggregate the Values using the normalized attention weights
    # V is (D, N), alpha is (N, N) -> output is (D, N)
    output = np.matmul(V, alpha)

    return output

# --- Testing the implementation ---
np.random.seed(42)

# Initialize distinct random weight matrices for Q, K, and V
W_q = np.random.normal(size=(D, D))
W_k = np.random.normal(size=(D, D))
W_v = np.random.normal(size=(D, D))

dot_attention_out = masked_dot_product_attention(X, W_q, W_k, W_v, A)

print("Dot-Product Graph Attention Output:")
print(np.round(dot_attention_out, 3))

Dot-Product Graph Attention Output:
[[ 1.809 -1.599 -0.52  -1.599  0.731  0.363  0.363  0.205]
 [ 2.     0.11  -0.087  0.123 -1.144 -3.191 -3.191  1.832]
 [-0.496  0.998  0.178  1.003 -2.057  1.587  1.587  0.352]
 [ 1.2   -1.594 -0.246 -1.599  3.713 -4.542 -4.542 -0.168]]
